##### ***子词嵌入***
###### 在英语中，helps和helped等系列单词都是help的变形形式，cat和cats之间的关系和dog与dogs之间的关系相同，而这种对词内部的结构探讨是word2vec和GloVe等模型都没有考虑的。所以我们需要一种新的模型来考虑这种关系。
##### ***fastText模型***
###### 在word2vec中，同一个词的不同变形形式直接由不同的向量表示，不需要共享参数，而fastText模型则考虑了这种关系，提出了一种子词嵌入方法，其中子词是一个字符n-gram，而fastText可以被任务是子词级别的跳元模型，其中每个中心词由其子词向量之和表示。<br>举例来说"where"这个单词作为中心词，首先在词的开头和末尾添加"<"和">"，以将前缀和后缀与其他子词区分开来。然后从词中提取字符n-gram，例如n=3，我们将获得长度为3的所有子词："<wh"，"whe"，"her"，"ere"，"re>"和特殊子词"<where>"。在该模型中，我们使用$\mathcal{G}_w$表示对于任意词$w$的长度在3和6之间的所有子词与其特殊子词的并集。词表是所有子词的集合，假设$z_g$是词典中的子词$g$，则跳元模型中作为中心词$w$的向量$v_w$是其子词向量的和:
$$v_w = \sum_{g\in\mathcal{G}_w}z_g$$
###### fastText模型的其余部分与跳元模型一致，也就是说fastText模型的优化点在于将中心词向量的表示换成了子词向量的和。如此一来它的词量会更大，模型参数也会更多。
##### ***字节对编码(Byte Pair Encoding, BPE)***
###### 在fastText模型中，所有提取的子词都必须是指定的长度，如3-6，因此词表大小不能预定义，因为不同的语料中有不同的词不同的序列，事先不知道最终会有多少种不同的子词，也就不能事先知道词表大小以用来分配模型参数，需要事先扫描全部语料进行计算，这就增加了预处理的步骤，包括之前的word2vec模型也是同样的问题。所以为了在固定大小的词表中获得可变长度的子词，我们可以应用一种称为字节对编码的压缩算法来提取子词。<br>具体来说BPE执行训练数据集的统计分析，即在单词内发现公共符号，从长度1的符号开始，BPE迭代地合并最频繁的连续符号对以产生新的更长的符号。同时为了提高效率，合并操作只在单词内部进行，不会把前一个单词的最后一个字符和后一个单词的第一个字符合并。
###### 通过构建一个raw_token_freqs字典，将词映射到该词在整个数据集中出现的频率，同时在每个词的后面加上一个特殊符号'_'来作为词边界的标记，而且由于仅从单个字符和特殊符号的词开始合并操作的，所以将每个词的每个字符间都插入空格，以此来作为单独的符号来处理，也就是把词表示成由空格分隔的符号序列。

In [1]:
import collections

symbols = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
           'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z',
           '_', '[UNK]']
raw_token_freqs = {'fast_': 4, 'faster_': 3, 'tall_': 5, 'taller_': 4}
token_freqs = {}
for token, freq in raw_token_freqs.items():
    token_freqs[' '.join(list(token))] = raw_token_freqs[token]
token_freqs

{'f a s t _': 4, 'f a s t e r _': 3, 't a l l _': 5, 't a l l e r _': 4}

In [2]:
# 接下来定义get_max_freq_pair函数，用来返回词内最频繁出现的连续符号对
def get_max_freq_pair(token_freqs):
    pairs = collections.defaultdict(int)
    for token, freq in token_freqs.items():
        symbols = token.split()
        for i in range(len(symbols) - 1):
            # "pairs"的键是两个连续符号的元组
            pairs[symbols[i], symbols[i+1]] += freq
    
    return max(pairs, key=pairs.get)

In [3]:
def merge_symbols(max_freq_pair, token_freqs, symbols):
    symbols.append(''.join(max_freq_pair))
    new_token_freqs = dict()
    for token, freq in token_freqs.items():
        new_token = token.replace(' '.join(max_freq_pair),
                                  ''.join(max_freq_pair))
        new_token_freqs[new_token] = token_freqs[token]
    return new_token_freqs

###### 接着我们对token_freqs迭代地执行BPE算法。

In [4]:
num_merges = 10 
for i in range(num_merges):
    max_freq_pair = get_max_freq_pair(token_freqs)
    token_freqs = merge_symbols(max_freq_pair, token_freqs, symbols)
    print(f'merge{i+1}:', max_freq_pair)

merge1: ('t', 'a')
merge2: ('ta', 'l')
merge3: ('tal', 'l')
merge4: ('f', 'a')
merge5: ('fa', 's')
merge6: ('fas', 't')
merge7: ('e', 'r')
merge8: ('er', '_')
merge9: ('tall', '_')
merge10: ('fast', '_')


In [6]:
print(symbols)
print(list(token_freqs.keys()))

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '_', '[UNK]', 'ta', 'tal', 'tall', 'fa', 'fas', 'fast', 'er', 'er_', 'tall_', 'fast_']
['fast_', 'fast er_', 'tall_', 'tall er_']


###### BPE算法的结果取决于正在使用的数据集，我们还可以使用从一个数据集学习到的子词来切分另一个数据集的单词。下面的segment_BPE函数尝试将单词从输入参数symbols中分成可能的最长子词。

In [ ]:
def segment_BPE(tokens, symbols):
    outputs = []
    for token in tokens:
        start, end = 0, len(token)
        cur_output = []
        # 具有symbols中可能最长子字的词元段
        while start < len(token) and start < end:
            if token[start: end] in symbols: # 如果当前子词在symbols中
                # 则将当前子词添加到输出中
                cur_output.append(token[start: end])
                # 更新start和end
                start = end 
                end = len(token)
            else:
                end -= 1
        # 经过贪心匹配后，如果start小于token的长度，说明有罕见字符或未知字符
        if start < len(token):
            cur_output.append('[UNK]')
        outputs.append(' '.join(cur_output))
    return outputs


In [8]:
tokens = ['tallest_', 'fatter_']
print(segment_BPE(tokens, symbols))

['tall e s t _', 'fa t t er_']
